# 🚔 Sri Lanka Police AI — Master Production Setup (v2.2)

This notebook provides a fully automated pipeline for:
1. **Fine-Tuning** Gemma-2-9B using Unsloth (Institutional Dataset).
2. **Exporting** to GGUF format for Ollama.
3. **Serving** a Production-Grade Flask API with Surya GPU OCR.

### ⚙️ Settings
- **Accelerator**: GPU T4 x2 (recommended) or P100.
- **Internet**: Must be ON.
- **Dataset**: Attach your `training_data.jsonl` to `/kaggle/input`.

In [ ]:
# CELL 1: Install Dependencies (Pinned for Stability)
# CRITICAL: Fixing NumPy 2.0 binary incompatibility error
!pip uninstall -y numpy
!pip install --quiet "numpy<2.0.0" "transformers>=4.43.0" "pyngrok==7.2.2" unsloth surya-ocr Pillow flask requests
!sudo apt-get install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

import numpy as np
import torch
print(f"[OK] NumPy: {np.__version__}")
print(f"[OK] PyTorch: {torch.__version__}")
print(f"[OK] CUDA: {torch.cuda.is_available()}")

In [ ]:
# CELL 2: Automated Fine-Tuning
import os, glob
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# Find dataset
DATA_SEARCH = glob.glob("/kaggle/input/**/training_data.jsonl", recursive=True)
if not DATA_SEARCH:
    print("⚠️ Training data not found. Skipping fine-tuning stage...")
else:
    TRAIN_PATH = DATA_SEARCH[0]
    print(f"[AI] Training using: {TRAIN_PATH}")
    
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/gemma-2-9b-it-bnb-4bit",
        max_seq_length = 2048, load_in_4bit = True
    )
    
    model = FastLanguageModel.get_peft_model(
        model, r = 32, lora_alpha = 16, lora_dropout = 0,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )
    
    dataset = load_dataset("json", data_files=TRAIN_PATH, split="train")
    dataset = dataset.map(lambda e: { "text" : [tokenizer.apply_chat_template(c, tokenize=False) for c in e["messages"]] }, batched=True)
    
    trainer = SFTTrainer(
        model = model, tokenizer = tokenizer, train_dataset = dataset, 
        dataset_text_field = "text", max_seq_length = 2048,
        args = TrainingArguments(
            per_device_train_batch_size = 2, gradient_accumulation_steps = 4,
            warmup_steps = 5, max_steps = 60, learning_rate = 2e-4, fp16 = True,
            logging_steps = 1, optim = "adamw_8bit", output_dir = "output"
        )
    )
    
    trainer.train()
    print("[SAVING] Exporting Fine-tuned model to GGUF...")
    model.save_pretrained_gguf("police-ai-master-gguf", tokenizer, quantization_method = "q4_k_m")
    print("[OK] Fine-tuning Complete.")

In [ ]:
# CELL 3: Start Ollama + Create Master Model
import subprocess, time, os, glob
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)

GGUF_DIR = "police-ai-master-gguf"
if os.path.exists(GGUF_DIR):
    print("[GO] Creating model from Fine-Tuned GGUF...")
    gguf_path = glob.glob(f"{GGUF_DIR}/*.gguf")[0]
    # Using the refined Modelfile parameters for institutional accuracy
    modelfile = f"""FROM {gguf_path}
PARAMETER temperature 0.01
PARAMETER repeat_penalty 1.5
PARAMETER num_ctx 32768
PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
SYSTEM """You are the Sovereign Sri Lanka Police AI Architect.
Your primary directive is Institutional Integrity and Factual Zero-Hallucination.
You ONLY output valid JSON. Never add explanation, markdown, or extra text.

OFFENCE CLASSIFICATION:
- Category 04: SERIOUS CRIMES COMMITTED (Includes Homicides)
- Category 10: FATAL ACCIDENTS
""""""
    with open("Modelfile", "w") as f: f.write(modelfile)
    !ollama create police-ai-master -f Modelfile
else:
    print("[FALLBACK] Falling back to Base Gemma-2-9B")
    !ollama pull gemma2:9b
    !ollama create police-ai-master -b gemma2:9b

In [ ]:
# CELL 4: Load Surya OCR
import torch, time, os
from PIL import Image
print("[AI] Loading Surya OCR into GPU...")

SURYA_LOADED = False
recognition_predictor = None
detection_predictor = None
surya_api_version = None

try:
    from surya.recognition import RecognitionPredictor
    from surya.detection import DetectionPredictor
    from surya.foundation import FoundationPredictor
    
    foundation_predictor = FoundationPredictor()
    recognition_predictor = RecognitionPredictor(foundation_predictor)
    detection_predictor = DetectionPredictor()
    SURYA_LOADED = True
    surya_api_version = "new-foundation"
    print("[OK] Surya OCR loaded successfully (New API)")
except Exception as e:
    print(f"⚠️ Surya OCR loading failed: {e}")

In [ ]:
# CELL 5: Flask API Server (Dual Logic: OCR + AI)
from flask import Flask, request, jsonify, Response
import threading, io, json, requests

app = Flask(__name__)
gpu_lock = threading.Lock()

@app.route("/health")
def health(): return jsonify({"status": "ok", "gpu": torch.cuda.is_available(), "ocr": SURYA_LOADED})

@app.route("/process", methods=["POST"])
def process():
    if "file" not in request.files: return jsonify({"error": "No file"}), 400
    image = Image.open(io.BytesIO(request.files["file"].read())).convert("RGB")
    
    # 1. OCR
    with gpu_lock:
        preds = recognition_predictor([image], det_predictor=detection_predictor)
        torch.cuda.empty_cache()
    
    ocr_text = "\n".join([line.text for line in preds[0].text_lines]) if hasattr(preds[0], "text_lines") else ""
    
    # 2. AI Extraction
    prompt = f"Extract all police report fields from the following OCR text and return ONLY valid JSON:\n\n{ocr_text}"
    resp = requests.post("http://localhost:11434/api/generate", json={
        "model": "police-ai-master", "prompt": prompt, "stream": False
    }).json()
    
    return jsonify({"ocr_text": ocr_text, "ai_json": json.loads(resp.get("response", "{}"))})

# Proxy routes for Ollama compatibility
@app.route("/api/generate", methods=["POST"])
def generate(): 
    resp = requests.post("http://localhost:11434/api/generate", json=request.json)
    return Response(resp.content, content_type="application/json")

def run_flask(): app.run(host="0.0.0.0", port=5050, debug=False, use_reloader=False)
threading.Thread(target=run_flask, daemon=True).start()
print("[OK] Flask Server started on port 5050")

In [ ]:
# CELL 6: Ngrok Tunnel + Keep-Alive
from pyngrok import ngrok
import time

NGROK_AUTH_TOKEN = "3C0C7gFkX4IQuTy2cMMPHYznbNh_4CZ5YG6ekExX6sBKNfhpv" # Use yours!
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)

url = ngrok.connect(5050, domain="nestable-mireya-uncarbonized.ngrok-free.dev").public_url
print(f"\n!! POLICE AI PUBLIC URL: {url}")
print(f"Update your config.json with: {url}/api/generate")

while True:
    time.sleep(30)
    print(f"Alive Server Active | {url}")